# Project: Intelligent Retail (Edge Model)
### Datum: 2026-03-15
### Team: Michal Kakol

## 0. Setup:

In [1]:
import logging
import os
import pandas as pd
from PIL import Image
import pickle
logging.basicConfig(level=logging.INFO)

## 1. Data Loading:

### 1.1. Edge:

In [2]:
def edge_load_images(image_dir):
    images = []
    for file in os.listdir(image_dir):
        if file.endswith(".jpg") or file.endswith(".png"):
            path = os.path.join(image_dir, file)
            img = Image.open(path)
            images.append(img)
    return images

images = edge_load_images("../data/raw/image/frames")

Loaded images: 2000


In [3]:
def edge_load_labels(path):
    df = pd.read_csv(path)
    labels = df["count"].tolist()
    return labels, df

labels, df_labels = edge_load_labels("../data/raw/image/labels.csv")
df_labels.head()

Loaded labels: 2000


,id,count
0,1,35
1,2,41
2,3,41
3,4,44
4,5,41


In [4]:
def edge_validate(images, labels):
    print("images:", len(images))
    print("labels:", len(labels))

edge_validate(images, labels)

Number of images: 2000
Number of labels: 2000


### 1.2. Cloud:

In [5]:
def cloud_load_data(path):
    return pd.read_csv(path)

df = cloud_load_data("../data/raw/text/train.csv")
df.head()

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


In [6]:
def cloud_validate(df):
    print("Shape:", df.shape)
    print("Null:", df.isnull().sum())
    print("Negatives:", (df["sales"] < 0).sum())

cloud_validate(df)

Shape: (913000, 4)
Missing values:
 date     0
store    0
item     0
sales    0
dtype: int64
Negative sales: 0
Null rows: 0


## 2. Data Processing:

### 2.1. Edge:

In [7]:
def edge_preprocess_images(images, size=(320, 240)):
    processed = []
    for img in images:
        img = img.resize(size)
        img = img.convert("RGB")
        processed.append(img)
    return processed

processed_images = edge_preprocess_images(images)

Processed images: 2000


In [8]:
def edge_save_data(data, path):
    with open(path, "wb") as f:
        pickle.dump(data, f)

edge_save_data(processed_images, "../data/processed/images.pkl")

### 2.2. Cloud:

In [9]:
def cloud_transform(df):
    df = df.dropna()
    df["date"] = pd.to_datetime(df["date"])
    df["day_of_week"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    return df

df_transformed = cloud_transform(df)
df_transformed.head()

,date,store,item,sales,day_of_week,month
0,2013-01-01,1,1,13,1,1
1,2013-01-02,1,1,11,2,1
2,2013-01-03,1,1,14,3,1
3,2013-01-04,1,1,13,4,1
4,2013-01-05,1,1,10,5,1


In [10]:
def cloud_save_data(df, path):
    df.to_parquet(path)

cloud_save_data(df_transformed, "../data/processed/sales.parquet")

In [11]:
def cloud_validate_output(path):
    df = pd.read_parquet(path)
    print("Validated shape:", df.shape)
    print(df.head())

cloud_validate_output("../data/processed/sales.parquet")

Validated shape: (913000, 6)
        date  store  item  sales  day_of_week  month
0 2013-01-01      1     1     13            1      1
1 2013-01-02      1     1     11            2      1
2 2013-01-03      1     1     14            3      1
3 2013-01-04      1     1     13            4      1
4 2013-01-05      1     1     10            5      1


## 3. Pipeline:

### 3.1. Edge:

In [17]:
def edge_pipeline():
    logging.info("edge pipeline start")

    images = edge_load_images("../data/raw/image/frames")
    labels, _ = edge_load_labels("../data/raw/image/labels.csv")
    edge_validate(images, labels)
    images = edge_preprocess_images(images)
    edge_save_data(images, "../data/processed/images.pkl")

    logging.info("edge pipeline end")

edge_pipeline()

INFO:root:Running edge pipeline


Number of images: 2000
Number of labels: 2000


INFO:root:Edge pipeline finished


### 3.1. Cloud:

In [16]:
def cloud_pipeline():
    logging.info("cloud pipeline start")

    df = cloud_load_data("../data/raw/text/train.csv")
    cloud_validate(df)
    df = cloud_transform(df)
    cloud_save_data(df, "../data/processed/sales.parquet")
    cloud_validate_output("../data/processed/sales.parquet")

    logging.info("cloud pipeline end")

cloud_pipeline()

INFO:root:Running cloud pipeline


Shape: (913000, 4)
Missing values:
 date     0
store    0
item     0
sales    0
dtype: int64
Negative sales: 0
Null rows: 0


INFO:root:Cloud pipeline finished


Validated shape: (913000, 6)
        date  store  item  sales  day_of_week  month
0 2013-01-01      1     1     13            1      1
1 2013-01-02      1     1     11            2      1
2 2013-01-03      1     1     14            3      1
3 2013-01-04      1     1     13            4      1
4 2013-01-05      1     1     10            5      1
